# Wikidata Article Importer

Created by [Matt Artz](https://www.mattartz.me/) | [GitHub](https://github.com/MattArtzAnthro) | [ORCID](https://orcid.org/0000-0002-3822-1429)

---

## What This Notebook Does

This notebook transforms bibliographic metadata from CSV exports into Wikidata-ready QuickStatements format. It checks for duplicates via the Wikidata Scholarly SPARQL endpoint, maps journal names to Q-identifiers, and generates batch upload files.

## Key Features

- **Scholarly Endpoint**: Uses query-scholarly.wikidata.org (since May 2025 graph split)
- **Smart Journal Matching**: Handles variations like '&' vs 'and'
- **QuickStatements V1 Output**: Uses PIPE separators for reliable parsing

## Workflow

1. **Upload**: Upload articles CSV and journal mapping CSV
2. **Process**: Check each DOI against Wikidata's scholarly endpoint
3. **Generate**: Create QuickStatements V1 file with PIPE separators
4. **Export**: Download QuickStatements file and summary CSV

## Citation

> Artz, M. (2026). Wikidata Tools. GitHub. https://github.com/MattArtzAnthro/wikidata-tools

*A citable DOI will be available via Zenodo.*

## License

[CC BY-NC 4.0](https://creativecommons.org/licenses/by-nc/4.0/)

## Setup

In [ ]:
!pip install requests pandas ipywidgets -q

import requests
import pandas as pd
import re
import time
from datetime import datetime
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets
from io import BytesIO

print("Setup complete.")

## Helper Functions

*Since May 2025, scholarly articles are ONLY on query-scholarly.wikidata.org*

In [ ]:
# IMPORTANT: Since May 2025 graph split, scholarly articles are ONLY on the scholarly endpoint
# The main endpoint (query.wikidata.org) no longer contains scholarly articles!
SCHOLARLY_ENDPOINT = "https://query-scholarly.wikidata.org/sparql"

def clean_doi(doi_input):
    if not doi_input or not isinstance(doi_input, str):
        return None
    doi_input = str(doi_input).strip()
    for pattern in [r'https?://(?:dx\.)?doi\.org/(.+)', r'doi:(.+)']:
        match = re.search(pattern, doi_input, re.IGNORECASE)
        if match:
            doi_input = match.group(1)
            break
    if re.match(r'^10\.\d+/.+', doi_input):
        return doi_input.strip()
    return None

def check_wikidata_for_doi(doi, verbose=False):
    """
    Check the SCHOLARLY endpoint for existing DOI.
    Since May 2025, scholarly articles are ONLY on query-scholarly.wikidata.org
    """
    doi_upper = doi.upper()
    sparql_query = f'SELECT ?item WHERE {{ ?item wdt:P356 "{doi_upper}" }}'

    try:
        if verbose:
            print(f"      Querying: {SCHOLARLY_ENDPOINT}")
            print(f"      DOI: {doi_upper}")

        response = requests.get(
            SCHOLARLY_ENDPOINT,
            params={"query": sparql_query, "format": "json"},
            headers={"User-Agent": "WikidataArticleImporter/1.0 (matt@mattartz.me)"},
            timeout=30
        )

        if verbose:
            print(f"      Status: {response.status_code}")

        response.raise_for_status()
        data = response.json()
        results = data.get("results", {}).get("bindings", [])

        if results:
            qid = results[0]["item"]["value"].split("/")[-1]
            if verbose:
                print(f"      Found: {qid}")
            return qid
        else:
            if verbose:
                print("      Not found in Wikidata")
            return None

    except requests.exceptions.Timeout:
        print(f"   TIMEOUT querying scholarly endpoint")
        return None
    except requests.exceptions.RequestException as e:
        print(f"   ERROR: {e}")
        return None
    except Exception as e:
        print(f"   UNEXPECTED ERROR: {e}")
        return None

def parse_authors(authors_string):
    if not authors_string or pd.isna(authors_string):
        return []
    return [a.strip() for a in str(authors_string).split(';') if a.strip()]

def parse_publication_date(date_string, year_string=None):
    if not date_string or pd.isna(date_string):
        if year_string and not pd.isna(year_string):
            return f"+{int(year_string)}-00-00T00:00:00Z", 9
        return None, None
    date_str = str(date_string).strip()
    if re.match(r'^\d{4}-\d{2}-\d{2}$', date_str):
        return f"+{date_str}T00:00:00Z", 11
    if re.match(r'^\d{4}-\d{2}$', date_str):
        return f"+{date_str}-00T00:00:00Z", 10
    if re.match(r'^\d{4}$', date_str):
        return f"+{date_str}-00-00T00:00:00Z", 9
    if year_string and not pd.isna(year_string):
        return f"+{int(year_string)}-00-00T00:00:00Z", 9
    return None, None

def get_first_issn(issn_string):
    if not issn_string or pd.isna(issn_string):
        return None
    issns = re.split(r'[;,]', str(issn_string))
    return issns[0].strip() if issns else None

print(f"Helper functions loaded.")
print(f"Using scholarly endpoint: {SCHOLARLY_ENDPOINT}")

## Test Endpoint Connection

*Run this cell to verify the scholarly endpoint is accessible.*

In [ ]:
# Test the scholarly endpoint with a known DOI
print("Testing scholarly endpoint connection...")
print()

test_doi = "10.1111/ANHU.12540"  # Known article
print(f"Testing with DOI: {test_doi}")

result = check_wikidata_for_doi(test_doi, verbose=True)

print()
if result:
    print(f"SUCCESS: Found {result}")
    print(f"View at: https://www.wikidata.org/wiki/{result}")
else:
    print("NOT FOUND or endpoint issue")
    print("If this DOI should exist, check your network connection.")

## Journal Q-ID Lookup

*Handles '&' vs 'and' and other variations.*

In [ ]:
journal_mapping = {}
journal_mapping_normalized = {}

def normalize_journal_name(name):
    """Normalize journal name for fuzzy matching."""
    if not name:
        return ""
    name = str(name).lower().strip()
    name = name.replace('&', 'and')
    name = re.sub(r'[^\w\s-]', '', name)
    name = ' '.join(name.split())
    return name

def load_journal_mapping(df):
    global journal_mapping, journal_mapping_normalized
    journal_mapping = {}
    journal_mapping_normalized = {}

    for _, row in df.iterrows():
        journal_url = str(row.get('journal', ''))
        if 'wikidata.org' in journal_url:
            qid = journal_url.split('/')[-1]
        else:
            qid = journal_url

        label = str(row.get('journalLabel', '')).strip()

        if qid and label:
            journal_mapping[label.lower()] = qid
            normalized = normalize_journal_name(label)
            journal_mapping_normalized[normalized] = qid

    return len(journal_mapping)

def lookup_journal_qid(journal_name):
    """Look up Q-ID with smart matching."""
    if not journal_name or pd.isna(journal_name):
        return None

    key = str(journal_name).strip().lower()
    if key in journal_mapping:
        return journal_mapping[key]

    normalized_key = normalize_journal_name(journal_name)
    if normalized_key in journal_mapping_normalized:
        return journal_mapping_normalized[normalized_key]

    for stored_name, qid in journal_mapping_normalized.items():
        if stored_name in normalized_key or normalized_key in stored_name:
            return qid

    return None

print("Journal lookup functions loaded.")

## QuickStatements Generator

*CRITICAL: Uses PIPE (|) separators. Spaces DO NOT work!*

In [ ]:
def generate_quickstatements(articles_to_add):
    """
    Generate QuickStatements V1 format using PIPE separators.

    CRITICAL: V1 format requires PIPE (|) or TAB separators!
    Spaces cause blank items to be created.

    Correct:  LAST|Len|"Title"
    WRONG:    LAST Len "Title"  <-- causes blank items!
    
    NOTE: Descriptions include volume, issue, and DOI to ensure uniqueness
    and avoid triggering Wikidata's abuse filter on batch imports.
    """
    qs_lines = []

    for article in articles_to_add:
        title = str(article.get('title', '')).replace('"', '""')[:250]
        if not title:
            continue

        qs_lines.append('CREATE')
        qs_lines.append(f'LAST|Len|"{title}"')

        # Description - include volume, issue, and DOI to ensure uniqueness
        # This prevents Wikidata's abuse filter from flagging identical descriptions
        pub_date = article.get('publication_date', '')
        journal = article.get('journal', '')
        volume = article.get('volume')
        issue = article.get('issue')
        doi = article.get('doi', '')
        
        year_match = re.search(r'\d{4}', str(pub_date)) if pub_date else None
        year_str = year_match.group() if year_match else ''
        
        if journal:
            desc = f"scholarly article published in {journal}"
            
            # Add volume and issue if available
            vol_str = str(volume).strip() if volume and not pd.isna(volume) else ''
            iss_str = str(issue).strip() if issue and not pd.isna(issue) else ''
            
            if vol_str and iss_str:
                desc += f", vol. {vol_str} no. {iss_str}"
            elif vol_str:
                desc += f", vol. {vol_str}"
            
            if year_str:
                desc += f" ({year_str})"
            
            # Add DOI to guarantee uniqueness
            if doi:
                desc += f" (DOI: {doi.upper()})"
        else:
            desc = "scholarly article"
            if year_str:
                desc += f" ({year_str})"
            if doi:
                desc += f" (DOI: {doi.upper()})"
        
        qs_lines.append(f'LAST|Den|"{desc[:250]}"')

        qs_lines.append('LAST|P31|Q13442814')
        qs_lines.append(f'LAST|P1476|en:"{title}"')

        if doi:
            qs_lines.append(f'LAST|P356|"{doi.upper()}"')

        time_str = article.get('time_string')
        precision = article.get('precision')
        if time_str and precision:
            qs_lines.append(f'LAST|P577|{time_str}/{precision}')

        # Journal
        journal_qid = article.get('journal_qid')
        if journal_qid:
            qs_lines.append(f'LAST|P1433|{journal_qid}')

        # Authors with ordinal qualifiers
        for idx, author in enumerate(article.get('authors', []), 1):
            author_clean = str(author).replace('"', '""')
            qs_lines.append(f'LAST|P2093|"{author_clean}"|P1545|"{idx}"')

        volume = article.get('volume')
        if volume and not pd.isna(volume) and str(volume).strip():
            qs_lines.append(f'LAST|P478|"{str(volume).strip()}"')

        issue = article.get('issue')
        if issue and not pd.isna(issue) and str(issue).strip():
            qs_lines.append(f'LAST|P433|"{str(issue).strip()}"')

        pages = article.get('pages')
        if pages and not pd.isna(pages) and str(pages).strip():
            qs_lines.append(f'LAST|P304|"{str(pages).strip()}"')

        issn = article.get('issn')
        if issn:
            qs_lines.append(f'LAST|P236|"{issn}"')

        url = article.get('url')
        if url and not pd.isna(url) and str(url).strip():
            qs_lines.append(f'LAST|P953|"{str(url).strip()}"')

        qs_lines.append('LAST|P407|Q1860')
        qs_lines.append('')

    return '\n'.join(qs_lines)

print("QuickStatements generator loaded (PIPE separators).")
print("Descriptions include vol/issue/DOI for uniqueness.")


## Data Upload Interface

In [ ]:
articles_df = None
journals_df = None

articles_upload = widgets.FileUpload(accept='.csv', multiple=False, description='Articles CSV')
journals_upload = widgets.FileUpload(accept='.csv', multiple=False, description='Journals CSV')
articles_output = widgets.Output()
journals_output = widgets.Output()

def on_articles_upload(change):
    global articles_df
    with articles_output:
        clear_output()
        if articles_upload.value:
            try:
                file_info = list(articles_upload.value.values())[0]
                articles_df = pd.read_csv(BytesIO(file_info['content']))
                print(f"Loaded {len(articles_df)} articles")
                if 'Journal' in articles_df.columns:
                    journals_found = articles_df['Journal'].dropna().unique()
                    print(f"Journals: {list(journals_found)[:3]}")
            except Exception as e:
                print(f"Error: {e}")

def on_journals_upload(change):
    global journals_df
    with journals_output:
        clear_output()
        if journals_upload.value:
            try:
                file_info = list(journals_upload.value.values())[0]
                journals_df = pd.read_csv(BytesIO(file_info['content']))
                count = load_journal_mapping(journals_df)
                print(f"Loaded {count} journal mappings")
                for label, qid in list(journal_mapping.items())[:3]:
                    print(f"  '{label}' -> {qid}")
            except Exception as e:
                print(f"Error: {e}")

articles_upload.observe(on_articles_upload, names='value')
journals_upload.observe(on_journals_upload, names='value')

display(widgets.HBox([
    widgets.VBox([widgets.Label('Articles CSV:'), articles_upload, articles_output]),
    widgets.VBox([widgets.Label('Journals CSV:'), journals_upload, journals_output])
]))
print("Upload your CSV files above.")

## Process Articles

*Checks scholarly endpoint for each DOI.*

In [ ]:
articles_to_add = []
articles_existing = []
articles_failed = []

process_button = widgets.Button(description='Process Articles', button_style='primary')
progress_bar = widgets.IntProgress(value=0, min=0, max=100, description='Progress:')
process_output = widgets.Output()

def process_articles(button):
    global articles_to_add, articles_existing, articles_failed
    articles_to_add, articles_existing, articles_failed = [], [], []

    with process_output:
        clear_output()
        if articles_df is None:
            print("Please upload articles CSV first.")
            return

        if not journal_mapping:
            print("WARNING: No journal mappings loaded!")
            print()

        print(f"Using endpoint: {SCHOLARLY_ENDPOINT}")
        print()

        total = len(articles_df)
        progress_bar.max = total
        print(f"Processing {total} articles...\n")

        for idx, row in articles_df.iterrows():
            progress_bar.value = idx + 1
            doi = clean_doi(row.get('DOI', ''))
            title = str(row.get('Title', ''))[:50]

            if not doi:
                print(f"[{idx+1}] NO DOI: {title}...")
                articles_failed.append({'title': title, 'reason': 'No DOI'})
                continue

            # Check scholarly endpoint
            existing_qid = check_wikidata_for_doi(doi)

            if existing_qid:
                print(f"[{idx+1}] EXISTS ({existing_qid}): {title}...")
                articles_existing.append({'title': title, 'doi': doi, 'qid': existing_qid})
            else:
                authors = parse_authors(row.get('Authors', ''))
                time_str, precision = parse_publication_date(row.get('Publication Date'), row.get('Year'))
                journal_name = row.get('Journal', '')
                journal_qid = lookup_journal_qid(journal_name)

                article_data = {
                    'title': row.get('Title', ''),
                    'doi': doi,
                    'authors': authors,
                    'time_string': time_str,
                    'precision': precision,
                    'publication_date': row.get('Publication Date', ''),
                    'journal': journal_name,
                    'journal_qid': journal_qid,
                    'volume': row.get('Volume'),
                    'issue': row.get('Issue'),
                    'pages': row.get('Page'),
                    'issn': get_first_issn(row.get('ISSN')),
                    'url': row.get('URL')
                }
                articles_to_add.append(article_data)
                j_status = f"J:{journal_qid}" if journal_qid else "NO-J"
                print(f"[{idx+1}] NEW ({j_status}): {title}...")

            # Rate limiting
            time.sleep(0.5)

        print()
        print("=" * 60)
        print("SUMMARY")
        print("=" * 60)
        print(f"Already in Wikidata: {len(articles_existing)}")
        print(f"New (to be added):   {len(articles_to_add)}")
        print(f"Failed (no DOI):     {len(articles_failed)}")

        if articles_to_add:
            with_j = len([a for a in articles_to_add if a.get('journal_qid')])
            print(f"With journal Q-ID:   {with_j}/{len(articles_to_add)}")

process_button.on_click(process_articles)
display(widgets.VBox([process_button, progress_bar, process_output]))

## Generate QuickStatements Output

In [ ]:
generate_button = widgets.Button(description='Generate QuickStatements', button_style='success')
generate_output = widgets.Output()

def generate_qs(button):
    with generate_output:
        clear_output()
        if not articles_to_add:
            print("No new articles to add. Run 'Process Articles' first.")
            return

        print(f"Generating QuickStatements for {len(articles_to_add)} articles...")
        print("Format: V1 with PIPE (|) separators")
        print()

        qs_text = generate_quickstatements(articles_to_add)

        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f"quickstatements_{timestamp}.txt"
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(qs_text)

        print(f"Saved: {filename}")
        print(f"Pipe separators: {qs_text.count('|')}")

        print()
        print("--- PREVIEW (first article) ---")
        first_block = qs_text.split('\n\n')[0] if '\n\n' in qs_text else qs_text[:500]
        print(first_block)

        print()
        print("--- UPLOAD INSTRUCTIONS ---")
        print("1. Go to: https://quickstatements.toolforge.org/")
        print("2. Log in with your Wikidata account")
        print("3. Click 'New batch'")
        print("4. Paste the file contents")
        print("5. Click 'Import V1 commands'")
        print("6. Review and click 'Run'")

        try:
            from google.colab import files
            files.download(filename)
        except:
            pass

generate_button.on_click(generate_qs)
display(widgets.VBox([generate_button, generate_output]))

## Export Summary

In [ ]:
export_button = widgets.Button(description='Export Summary CSV', button_style='info')
export_output = widgets.Output()

def export_summary(button):
    with export_output:
        clear_output()
        rows = []
        for a in articles_existing:
            rows.append({'Status': 'EXISTS', 'Title': a.get('title',''), 'DOI': a.get('doi',''), 'QID': a.get('qid','')})
        for a in articles_to_add:
            rows.append({'Status': 'NEW', 'Title': a.get('title',''), 'DOI': a.get('doi',''), 'QID': ''})
        for a in articles_failed:
            rows.append({'Status': 'FAILED', 'Title': a.get('title',''), 'DOI': '', 'QID': ''})

        if not rows:
            print("No results.")
            return

        df = pd.DataFrame(rows)
        filename = f"summary_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        df.to_csv(filename, index=False)
        print(f"Exported: {filename} ({len(rows)} rows)")
        print(f"  EXISTS: {len(articles_existing)}")
        print(f"  NEW: {len(articles_to_add)}")
        print(f"  FAILED: {len(articles_failed)}")
        try:
            from google.colab import files
            files.download(filename)
        except:
            pass

export_button.on_click(export_summary)
display(widgets.VBox([export_button, export_output]))